In [9]:
import glob         #used to retrieve file paths using pattern matching (like finding all part-* files).
import os           #used for interacting with the operating system (paths, directories etc.).
import json         #used to read schema information stored in JSON format.
import pandas as pd #used for reading CSV data and converting it into DataFrame for processing.

In [10]:
def get_column_name(schemas, df_name, sorting_key = 'column_position'):

    # Extract column details of the specific dataset from schemas.json
    column_details = schemas[df_name]

    # Sort columns according to column_position so that columns appear in correct order
    column = sorted(column_details, key=lambda col: col[sorting_key])

    # Extract only the column names and return them as a list
    return [col['column_name'] for col in column]

In [11]:
def read_csv(file,schemas):

    # Extract dataset name from the file path
    # Example:
    # retail_db/orders/part-00000.txt → orders
    ds_name = os.path.basename(os.path.dirname(file))

    # Get column names for that dataset using schema
    columns = get_column_name(schemas, ds_name)

    # Read the CSV file into pandas dataframe using the extracted column names
    df = pd.read_csv(file, names = columns)

    # Return the dataframe
    return df

In [12]:
def to_json(df, base_dir, ds_name, file_name):

    # Create output file path
    # Example:
    # retail_db_json/orders/part-00000.json
    json_file_path = f'{base_dir}/{ds_name}/{file_name}'

    # Create directory if it does not exist
    os.makedirs(f'{base_dir}/{ds_name}', exist_ok = True)

    # Convert dataframe to JSON format
    # orient='records' → each row becomes a JSON object
    # lines=True → each JSON record written on a separate line
    df.to_json(json_file_path, orient = 'records', lines = True)

In [13]:
def file_converter(src_base_dir,base_dir,ds_name):

    # Load schema file which contains column definitions
    schemas = json.load(open("D:/Khushi/course/python/mini project/retail_db/schemas.json"))

    # Find all files that match pattern "part-*"
    # Example:
    # part-00000.txt
    files = glob.glob(f'{src_base_dir}/{ds_name}/part-*')

    # Loop through each file
    for file in files:

        # Print file currently being processed
        print(f'Processing {file}')

        # Read the file into dataframe
        df = read_csv(file,schemas)

        # Extract filename and change extension from .txt → .json
        file_name = os.path.basename(file).replace(".txt",".json")

        # Convert dataframe to JSON format
        to_json(df, base_dir, ds_name, file_name)

In [14]:
def process_file(ds_name = None):

    # Source directory where raw dataset exists
    src_base_dir = 'retail_db'

    # Destination directory where JSON files will be stored
    base_dir = 'retail_db_json'

    # Load schema file
    schemas = json.load(open("D:/Khushi/course/python/mini project/retail_db/schemas.json"))

    # If dataset name is not provided → process all datasets
    if not ds_name:
        ds_names = schemas.keys()

    # If dataset name is provided → process only that dataset
    else:
        ds_names = [ds_name]

    # Loop through all datasets
    for name in ds_names:

        # Print dataset name being processed
        print(f'Processing {name}')

        # Convert files of that dataset
        file_converter(src_base_dir, base_dir, name)

In [15]:
process_file()

# Run the complete pipeline
# This will read all datasets from retail_db
# Convert them to JSON format
# Save them in retail_db_json folder

Processing departments
Processing retail_db/departments\part-00000.txt
Processing categories
Processing retail_db/categories\part-00000.txt
Processing orders
Processing retail_db/orders\part-00000.txt
Processing products
Processing retail_db/products\part-products.txt
Processing customers
Processing retail_db/customers\part-000001.txt
Processing order_items
Processing retail_db/order_items\part-order_items.txt
